# Essay #8 — Does the Model Know Where It Looks?
## Attention Entropy as a Test of Failure Awareness

This notebook tests whether last-layer `[CLS]` attention entropy provides useful failure 

awareness in a small Vision Transformer trained on CIFAR-10.

The experiment does **not** claim that attention is an explanation, that attention reveals reasoning, 

or that Vision Transformers are generally better than CNNs. It asks one narrower question:

> Does attention entropy help the model recognize when its prediction is becoming unreliable?

Earlier Essay #6 and Essay #7 values are included only as contextual references. They were obtained with 

different models and attack protocols and are not a controlled head-to-head benchmark.

In [ ]:
# =============================================================
# Part A — Reproducible setup and experiment configuration
# =============================================================
import os
import json
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = './data'
CHECKPOINT_PATH = 'checkpoint_vit_cifar10_attention.pth'
RESULTS_PATH = 'essay8_attention_results_corrected.json'

BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.05
TARGET_CLEAN_DEFERRAL = 0.25
PGD_EPSILON = 0.03
PGD_STEP_SIZE = 0.007
PGD_STEPS = 40

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print(f'Device: {DEVICE}')
print(f'Seed: {SEED}')
print(f'Attack: white-box PGD | epsilon={PGD_EPSILON} | step={PGD_STEP_SIZE} | steps={PGD_STEPS}')


## Part B — CIFAR-10 and held-out calibration/evaluation splits

The training set is used only for fitting. The test set is split into a 2,000-image calibration split and 

an 8,000-image untouched evaluation split. The attention threshold is learned only from the calibration split.

In [ ]:
# =============================================================
# Part B — Dataset loading
# =============================================================
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_dataset = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=test_transform)
split_generator = torch.Generator().manual_seed(SEED)
calibration_dataset, evaluation_dataset = random_split(
    test_dataset, [2000, 8000], generator=split_generator
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
calibration_loader = DataLoader(calibration_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
evaluation_loader = DataLoader(evaluation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

CIFAR_MEAN_T = torch.tensor(CIFAR_MEAN, device=DEVICE).view(1, 3, 1, 1)
CIFAR_STD_T = torch.tensor(CIFAR_STD, device=DEVICE).view(1, 3, 1, 1)
CIFAR_MIN = (0.0 - CIFAR_MEAN_T) / CIFAR_STD_T
CIFAR_MAX = (1.0 - CIFAR_MEAN_T) / CIFAR_STD_T

def clamp_valid(x):
    return torch.max(torch.min(x, CIFAR_MAX), CIFAR_MIN)

print(f'Training samples: {len(train_dataset):,}')
print(f'Calibration samples: {len(calibration_dataset):,}')
print(f'Evaluation samples: {len(evaluation_dataset):,}')


## Part C — Small Vision Transformer

A 32×32 image is divided into a grid of 8×8 patches. Each patch is 4×4 pixels, so the sequence contains

64 image patches plus one `[CLS]` token. The model returns per-head attention explicitly so the entropy calculation is faithful to the implementation.

In [ ]:
# =============================================================
# Part C — Small ViT with explicit per-head attention
# =============================================================
class ViTBlock(nn.Module):
    def __init__(self, dim=128, heads=4, mlp_dim=256, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(mlp_dim, dim), nn.Dropout(dropout)
        )

    def forward(self, x, return_attention=False):
        x_norm = self.norm1(x)
        attn_out, attn_weights = self.attn(
            x_norm, x_norm, x_norm, need_weights=return_attention,
            average_attn_weights=False
        )
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, attn_weights

class SmallViT(nn.Module):
    def __init__(self, image_size=32, patch_size=4, num_classes=10,
                 dim=128, depth=4, heads=4, mlp_dim=256, dropout=0.1):
        super().__init__()
        assert image_size % patch_size == 0
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim = 3 * patch_size * patch_size
        self.patch_embed = nn.Linear(patch_dim, dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches + 1, dim) * 0.02)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([ViTBlock(dim, heads, mlp_dim, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def patchify(self, x):
        b, c, _, _ = x.shape
        p = self.patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)
        patches = patches.contiguous().view(b, c, -1, p * p)
        patches = patches.permute(0, 2, 1, 3).contiguous()
        return patches.view(b, -1, c * p * p)

    def forward(self, x, return_attention=False):
        b = x.size(0)
        tokens = self.patch_embed(self.patchify(x))
        cls = self.cls_token.expand(b, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.pos_embed
        tokens = self.dropout(tokens)
        attentions = []
        for block in self.blocks:
            tokens, attention = block(tokens, return_attention=return_attention)
            if return_attention:
                attentions.append(attention)
        logits = self.head(self.norm(tokens)[:, 0])
        return (logits, attentions) if return_attention else logits

vit_model = SmallViT().to(DEVICE)
with torch.no_grad():
    test_logits, test_attentions = vit_model(torch.randn(2, 3, 32, 32, device=DEVICE), return_attention=True)
print('Logits:', tuple(test_logits.shape))
print('Layers:', len(test_attentions))
print('Per-head attention shape:', tuple(test_attentions[-1].shape))


In [ ]:
# =============================================================
# Part D — Training and clean accuracy
# =============================================================
def train_vit(model, loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(loader, desc=f'Epoch {epoch + 1}/{EPOCHS}'):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        print(f'Epoch {epoch + 1}: loss={running_loss / len(loader):.4f}, lr={scheduler.get_last_lr()[0]:.6f}')
    return model

def evaluate_accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            correct += (logits.argmax(1).cpu() == labels).sum().item()
            total += len(labels)
    return 100.0 * correct / total

if os.path.exists(CHECKPOINT_PATH):
    vit_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE)['model_state_dict'])
    print('Loaded checkpoint:', CHECKPOINT_PATH)
else:
    print('Training ViT from scratch...')
    train_vit(vit_model, train_loader)
    torch.save({'model_state_dict': vit_model.state_dict()}, CHECKPOINT_PATH)
    print('Saved checkpoint:', CHECKPOINT_PATH)

clean_accuracy = evaluate_accuracy(vit_model, evaluation_loader)
print(f'ViT clean accuracy: {clean_accuracy:.2f}%')


## Part E — Last-layer `[CLS]` attention entropy

The score used in this notebook is not a general explanation score. It is the entropy of the last-layer `[CLS]` attention 

distribution over the 64 image patches, after averaging across heads. High entropy means a more diffuse distribution; 

low entropy means a more concentrated distribution.

In [ ]:
# =============================================================
# Part E — Attention extraction and entropy
# =============================================================
def attention_entropy(attentions, layer=-1, include_cls=False):
    # attention shape: [batch, heads, sequence, sequence]
    attn = attentions[layer]
    cls_attention = attn[:, :, 0, :]
    if not include_cls:
        cls_attention = cls_attention[:, :, 1:]
    mean_attention = cls_attention.mean(dim=1)
    mean_attention = mean_attention / (mean_attention.sum(dim=-1, keepdim=True) + 1e-12)
    return -(mean_attention * (mean_attention + 1e-12).log()).sum(dim=-1)

with torch.no_grad():
    sample_images, _ = next(iter(evaluation_loader))
    _, sample_attentions = vit_model(sample_images[:4].to(DEVICE), return_attention=True)
    sample_entropy = attention_entropy(sample_attentions)
print('Sample last-layer entropy:', sample_entropy.cpu().numpy())


In [ ]:
# =============================================================
# Part F — White-box FGSM and PGD attacks
# =============================================================
def fgsm_attack(model, images, labels, epsilon=PGD_EPSILON):
    model.eval()
    x = images.detach().clone().to(DEVICE).requires_grad_(True)
    loss = nn.CrossEntropyLoss()(model(x), labels.to(DEVICE))
    model.zero_grad(set_to_none=True)
    loss.backward()
    return clamp_valid(x + epsilon * x.grad.sign()).detach()

def pgd_attack(model, images, labels, epsilon=PGD_EPSILON, step_size=PGD_STEP_SIZE, steps=PGD_STEPS):
    model.eval()
    original = images.detach().clone().to(DEVICE)
    adversarial = original.clone()
    for _ in range(steps):
        adversarial.requires_grad_(True)
        loss = nn.CrossEntropyLoss()(model(adversarial), labels.to(DEVICE))
        model.zero_grad(set_to_none=True)
        loss.backward()
        with torch.no_grad():
            adversarial = adversarial + step_size * adversarial.grad.sign()
            delta = torch.clamp(adversarial - original, -epsilon, epsilon)
            adversarial = clamp_valid(original + delta)
    return adversarial.detach()


## Part G — Calibration and selective evaluation

The entropy threshold is calibrated on clean calibration images toward 25% deferral. The resulting evaluation 

deferral may differ slightly because the threshold is applied to a separate split. Risk is measured only on non-deferred predictions.

In [ ]:
# =============================================================
# Part G — Collect scores, calibrate, and evaluate
# =============================================================
def collect_outputs(loader, attack=None):
    scores, correct, predictions, labels = [], [], [], []
    vit_model.eval()
    for images, batch_labels in tqdm(loader, desc=f'Collecting {attack or "clean"}'):
        images = images.to(DEVICE)
        batch_labels = batch_labels.to(DEVICE)
        attacked_images = images if attack is None else pgd_attack(vit_model, images, batch_labels)
        with torch.no_grad():
            logits, attentions = vit_model(attacked_images, return_attention=True)
            batch_predictions = logits.argmax(1)
            batch_scores = attention_entropy(attentions)
        scores.append(batch_scores.cpu())
        correct.append((batch_predictions == batch_labels).cpu())
        predictions.append(batch_predictions.cpu())
        labels.append(batch_labels.cpu())
    return {
        'scores': torch.cat(scores).numpy(),
        'correct': torch.cat(correct).numpy().astype(bool),
        'predictions': torch.cat(predictions).numpy(),
        'labels': torch.cat(labels).numpy(),
    }

def calibrate_threshold(scores, target_deferral=TARGET_CLEAN_DEFERRAL):
    return float(np.quantile(scores, 1.0 - target_deferral))

def selective_metrics(scores, correct, threshold):
    deferred = scores > threshold
    predicted = ~deferred
    coverage = float(predicted.mean())
    deferral = float(deferred.mean())
    accuracy = float(correct[predicted].mean()) if predicted.any() else 0.0
    return {
        'coverage': 100.0 * coverage,
        'deferral_rate': 100.0 * deferral,
        'accuracy_on_predicted': accuracy * 100.0,
        'risk_on_predicted': (1.0 - accuracy) * 100.0,
        'n_predicted': int(predicted.sum()),
        'n_deferred': int(deferred.sum()),
        'total': int(len(scores)),
    }

calibration = collect_outputs(calibration_loader)
threshold = calibrate_threshold(calibration['scores'])
clean = collect_outputs(evaluation_loader)
adversarial = collect_outputs(evaluation_loader, attack='pgd')
clean_metrics = selective_metrics(clean['scores'], clean['correct'], threshold)
adversarial_metrics = selective_metrics(adversarial['scores'], adversarial['correct'], threshold)

print(f'Calibrated threshold: {threshold:.6f}')
print('Calibration:', selective_metrics(calibration['scores'], calibration['correct'], threshold))
print('Clean:', clean_metrics)
print('Adversarial:', adversarial_metrics)


## Part H — Diagnostic analysis

These diagnostics test whether entropy actually tracks failure, rather than relying only on one thresholded risk number. 

Clean and adversarial arrays are aligned by image, so the entropy change is paired per sample.

In [ ]:
# =============================================================
# Part H — Diagnostics and publication figures
# =============================================================
def safe_auc(scores, incorrect):
    return float(roc_auc_score(incorrect.astype(int), scores)) if len(np.unique(incorrect)) == 2 else float('nan')

def risk_coverage_curve(scores, correct):
    # Defer highest entropy first. The remaining predictions are accepted.
    order = np.argsort(-scores)
    accepted_correct = correct[order][::-1].cumsum()[::-1]
    accepted_count = np.arange(len(correct), 0, -1)
    coverage = accepted_count / len(correct)
    risk = 1.0 - accepted_correct / accepted_count
    return coverage, risk

clean_incorrect = ~clean['correct']
adv_incorrect = ~adversarial['correct']
entropy_delta = adversarial['scores'] - clean['scores']

diagnostics = {
    'clean_entropy_mean': float(clean['scores'].mean()),
    'adversarial_entropy_mean': float(adversarial['scores'].mean()),
    'mean_entropy_delta_adv_minus_clean': float(entropy_delta.mean()),
    'median_entropy_delta_adv_minus_clean': float(np.median(entropy_delta)),
    'clean_error_auroc': safe_auc(clean['scores'], clean_incorrect),
    'adversarial_error_auroc': safe_auc(adversarial['scores'], adv_incorrect),
    'clean_correct_mean_entropy': float(clean['scores'][clean['correct']].mean()),
    'clean_incorrect_mean_entropy': float(clean['scores'][clean_incorrect].mean()),
    'adversarial_correct_mean_entropy': float(adversarial['scores'][adversarial['correct']].mean()),
    'adversarial_incorrect_mean_entropy': float(adversarial['scores'][adv_incorrect].mean()),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(clean['scores'], bins=40, alpha=0.65, label='Clean', density=True)
axes[0, 0].hist(adversarial['scores'], bins=40, alpha=0.65, label='PGD adversarial', density=True)
axes[0, 0].axvline(threshold, color='black', linestyle='--', label=f'Threshold={threshold:.3f}')
axes[0, 0].set_title('Attention entropy distributions')
axes[0, 0].set_xlabel('Last-layer [CLS] entropy')
axes[0, 0].set_ylabel('Density')
axes[0, 0].legend()

axes[0, 1].hist(entropy_delta, bins=40, color='#C98B2E', alpha=0.85)
axes[0, 1].axvline(0, color='black', linestyle='--')
axes[0, 1].set_title('Paired entropy change under PGD')
axes[0, 1].set_xlabel('Adversarial entropy − clean entropy')
axes[0, 1].set_ylabel('Images')

coverage_clean, risk_clean = risk_coverage_curve(clean['scores'], clean['correct'])
coverage_adv, risk_adv = risk_coverage_curve(adversarial['scores'], adversarial['correct'])
axes[1, 0].plot(coverage_clean * 100, risk_clean * 100, label='Clean')
axes[1, 0].plot(coverage_adv * 100, risk_adv * 100, label='PGD adversarial')
axes[1, 0].set_title('Risk–coverage curve')
axes[1, 0].set_xlabel('Coverage (%)')
axes[1, 0].set_ylabel('Risk (%)')
axes[1, 0].invert_xaxis()
axes[1, 0].legend()

groups = [
    clean['scores'][clean['correct']], clean['scores'][clean_incorrect],
    adversarial['scores'][adversarial['correct']], adversarial['scores'][adv_incorrect]
]
axes[1, 1].boxplot(groups, labels=['Clean\ncorrect', 'Clean\nincorrect', 'PGD\ncorrect', 'PGD\nincorrect'])
axes[1, 1].set_title('Entropy by prediction correctness')
axes[1, 1].set_ylabel('Entropy')

plt.tight_layout()
plt.savefig('essay8_attention_diagnostics.png', dpi=220, bbox_inches='tight')
plt.show()

print(json.dumps(diagnostics, indent=2))


## Part I — Contextual reference to Essays #6 and #7

The following values are reproduced from the earlier essays for narrative context. They are not a controlled 

head-to-head benchmark because the models and attack protocols differ.

In [ ]:
# =============================================================
# Part I — Contextual comparison only
# =============================================================
contextual_references = [
    ('Deep Ensembles', 'Essay #7', 15.42, 10.31),
    ('MC Dropout', 'Essay #7', 17.15, 12.51),
    ('Multi-head disagreement', 'Essay #6', 19.86, None),
    ('Evidential evidence', 'Essay #6', 21.31, None),
    ('SWAG', 'Essay #7', 25.95, 20.94),
    ('Boundary distance', 'Essay #6', 77.06, None),
    ('Confidence', 'Essay #6', 78.70, None),
]
print('Contextual reference — not a controlled head-to-head benchmark')
print(f"{'Method':<28} {'Source':<12} {'Adv risk':>10} {'Clean risk':>12}")
print('-' * 66)
for name, source, adv_risk, clean_risk in contextual_references:
    clean_text = '—' if clean_risk is None else f'{clean_risk:.2f}%'
    print(f'{name:<28} {source:<12} {adv_risk:>9.2f}% {clean_text:>12}')
print(f"{'Attention entropy':<28} {'Essay #8':<12} {adversarial_metrics['risk_on_predicted']:>9.2f}% {clean_metrics['risk_on_predicted']:>12.2f}%")


In [ ]:
# =============================================================
# Part J — Clean machine-readable report and conclusion
# =============================================================
report = {
    'question': 'Does attention entropy provide useful failure awareness in a small Vision Transformer?',
    'scope': 'Last-layer [CLS] attention entropy, head-averaged over 64 image patches.',
    'dataset': 'CIFAR-10',
    'training_samples': len(train_dataset),
    'calibration_samples': len(calibration_dataset),
    'evaluation_samples': len(evaluation_dataset),
    'model': {
        'architecture': 'Small Vision Transformer',
        'patch_size': '4x4 pixels',
        'patch_count': 64,
        'embedding_dimension': 128,
        'layers': 4,
        'heads': 4,
        'mlp_dimension': 256,
        'clean_accuracy_percent': clean_accuracy,
    },
    'attack': {
        'type': 'white-box PGD against the ViT',
        'epsilon': PGD_EPSILON,
        'step_size': PGD_STEP_SIZE,
        'steps': PGD_STEPS,
    },
    'threshold': threshold,
    'target_clean_deferral_percent': TARGET_CLEAN_DEFERRAL * 100.0,
    'calibration': selective_metrics(calibration['scores'], calibration['correct'], threshold),
    'clean': clean_metrics,
    'adversarial': adversarial_metrics,
    'diagnostics': diagnostics,
    'interpretation': (
        'In this small ViT and under this white-box PGD attack, last-layer [CLS] attention entropy '
        'did not provide useful failure awareness. Its mean value and error-discrimination ability '
        'must be interpreted from the diagnostic values above; the result is scoped to this model, '
        'signal, dataset, and attack protocol.'
    ),
}

with open(RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

print('\n' + '=' * 72)
print('ESSAY #8 — FINAL REPORT')
print('=' * 72)
print(f"Question: {report['question']}")
print(f"ViT clean accuracy: {clean_accuracy:.2f}%")
print(f"Threshold: {threshold:.6f}")
print(f"Clean: risk={clean_metrics['risk_on_predicted']:.2f}%, coverage={clean_metrics['coverage']:.2f}%, deferral={clean_metrics['deferral_rate']:.2f}%")
print(f"PGD:   risk={adversarial_metrics['risk_on_predicted']:.2f}%, coverage={adversarial_metrics['coverage']:.2f}%, deferral={adversarial_metrics['deferral_rate']:.2f}%")
print(f"Mean entropy clean: {diagnostics['clean_entropy_mean']:.6f}")
print(f"Mean entropy PGD:   {diagnostics['adversarial_entropy_mean']:.6f}")
print(f"Mean entropy shift: {diagnostics['mean_entropy_delta_adv_minus_clean']:.6f}")
print(f"Clean error AUROC:  {diagnostics['clean_error_auroc']:.4f}")
print(f"PGD error AUROC:    {diagnostics['adversarial_error_auroc']:.4f}")
print('\nInterpretation:')
print(report['interpretation'])
print(f"\nSaved: {RESULTS_PATH}")
print('Saved: essay8_attention_diagnostics.png')
